# <font color="#418FDE" size="6.5" uppercase>**CV Splitters**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Use scikit-learn cross-validation utilities to estimate model performance. 
- Choose appropriate splitters for classification, regression, grouped, and time-ordered data. 
- Inspect fold assignments and predictions produced by cross-validation. 


## **1. Cross Validation Utilities**

### **1.1. Holdout Evaluation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_01_01.jpg?v=1784032674" width="250">



>* Split data into training and testing sets
>* Estimate performance on unseen examples

>* Fast, simple first evaluation method
>* Choose test size to balance reliability

>* Control randomness for reproducible holdout splits
>* Stratify imbalanced classes; prefer cross-validation later



In [ ]:
#@title Python Code - Holdout Evaluation

# This script demonstrates holdout evaluation.
# It splits data before model training.
# It reports performance on unseen examples.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Check that features and labels match.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Stratification keeps class proportions similar in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Scaling is fitted only on training data inside the pipeline.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)

# Train once on the training portion only.
model.fit(X_train, y_train)

# Evaluate only on examples held out from training.
test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

# Compare with training accuracy to discuss generalization.
train_accuracy = model.score(X_train, y_train)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Training examples: {len(X_train)}")
print(f"Held-out test examples: {len(X_test)}")
print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Holdout test accuracy: {test_accuracy:.3f}")



### **1.2. Cross Validation Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_01_02.jpg?v=1784032676" width="250">



>* Estimate performance across multiple data folds
>* Score variation reveals model reliability

>* Distinguish real skill from lucky splits
>* Compare average performance and fold consistency

>* Match scoring metrics to modeling goals
>* Compare models, but interpret estimates carefully



In [ ]:
#@title Python Code - Cross Validation Scores

# Estimate model performance with cross validation.
# Each fold gives one accuracy score.
# The output summarizes typical and stable performance.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Build a simple model with scaling inside each fold.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Stratified folds keep class proportions similar in each split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

# Summarize the fold scores as performance and stability.
rounded_scores = np.round(scores, 3)
mean_score = round(float(scores.mean()), 3)
std_score = round(float(scores.std()), 3)

print("scikit-learn version:", sklearn.__version__)
print("Fold accuracy scores:", rounded_scores)
print("Mean accuracy:", mean_score)
print("Score standard deviation:", std_score)



### **1.3. Cross Validate Predictions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_01_03.jpg?v=1784032677" width="250">



>* Predict held-out samples across folds
>* Inspect realistic case-level model behavior

>* Link performance scores to specific prediction errors
>* Compare held-out predictions with true outcomes

>* Use for evaluation, not final deployment
>* Inspect case-level behavior before final training



In [ ]:
#@title Python Code - Cross Validate Predictions

# Demonstrate cross-validated predictions for held-out samples.
# Each prediction comes from a different training fold.
# Compare fair predictions with true iris labels.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_predict

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate that features and labels align.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Use stratified folds to preserve class balance.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=200, random_state=42)

# Predict every sample only when it is held out.
cv_predictions = cross_val_predict(model, X, y, cv=cv)
cv_accuracy = accuracy_score(y, cv_predictions)

# Show a few case-level predictions in original order.
preview = pd.DataFrame(
    {
        "true": iris.target_names[y[:5]],
        "cv_pred": iris.target_names[cv_predictions[:5]],
    }
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Cross-validated accuracy: {cv_accuracy:.3f}")
print("First five held-out predictions:")
print(preview.to_string(index=False))
print("Each row was predicted by a model that did not train on it.")



## **2. Common CV Splitters**

### **2.1. KFold StratifiedKFold**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_02_01.jpg?v=1784032668" width="250">



>* Split data into rotating training-validation folds
>* Check performance beyond one lucky split

>* Use KFold for balanced regression or classification
>* Use StratifiedKFold for imbalanced class proportions

>* Use KFold for continuous targets
>* Use stratified, grouped, or time-aware splits



In [ ]:
#@title Python Code - KFold StratifiedKFold

# Compare ordinary and stratified cross-validation folds.
# Class balance matters for classification validation splits.
# The output shows fold class proportions clearly.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# Create a small imbalanced classification dataset.
features, target = make_classification(
    n_samples=60,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    weights=[0.85, 0.15],
    random_state=42,
)

# Validate that both classes are present.
class_counts = np.bincount(target)
if len(class_counts) != 2 or min(class_counts) == 0:
    raise ValueError("Expected two non-empty classes.")

# Build two splitters with the same fold count.
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Summarize the positive-class percentage in each validation fold.
def fold_positive_rates(splitter, splitter_name):
    rates = []
    for train_index, valid_index in splitter.split(features, target):
        valid_target = target[valid_index]
        rates.append(round(100 * np.mean(valid_target), 1))
    return splitter_name + ": " + str(rates)

# Print concise results for beginner comparison.
print("scikit-learn version: " + sklearn.__version__)
print("Overall positive class: " + str(round(100 * np.mean(target), 1)) + "%")
print(fold_positive_rates(kfold, "KFold validation positives (%)"))
print(fold_positive_rates(stratified, "StratifiedKFold positives (%)"))



### **2.2. Repeated Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_02_02.jpg?v=1784032670" width="250">



>* Repeat K-fold with new random splits
>* Average results for more stable estimates

>* Shows performance consistency across many splits
>* Gives dependable estimates for smaller datasets

>* Match repeated CV to data type
>* Use group or time-aware splitters when needed



In [ ]:
#@title Python Code - Repeated Cross Validation

# This example compares repeated stratified validation scores.
# Repetition reshuffles folds to reveal score variability.
# The output summarizes accuracy across repeated folds.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small imbalanced classification dataset.
features, target = make_classification(
    n_samples=240,
    n_features=8,
    n_informative=4,
    n_redundant=0,
    weights=[0.75, 0.25],
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Build one simple model with scaling inside each fold.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)

# Repeat three-fold stratified cross validation four times.
splitter = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=4,
    random_state=42,
)

# Each score comes from one validation fold.
scores = cross_val_score(
    model,
    features,
    target,
    cv=splitter,
    scoring="accuracy",
)

# Summarize both average performance and variability.
mean_score = np.mean(scores)
std_score = np.std(scores)
rounded_scores = np.round(scores, 3)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Repeated folds evaluated: {len(scores)}")
print(f"Accuracy scores: {rounded_scores}")
print(f"Mean accuracy: {mean_score:.3f}")
print(f"Score standard deviation: {std_score:.3f}")



### **2.3. Shuffle Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_02_03.jpg?v=1784032672" width="250">



>* Repeated random train-test splits
>* Control split count and test size

>* Best for independent, unordered regression data
>* Avoid imbalanced classes or grouped observations

>* Random splits reveal performance sensitivity
>* Avoid Shuffle Split for time-ordered data



In [ ]:
#@title Python Code - Shuffle Split

# Demonstrate ShuffleSplit on independent regression data.
# Compare repeated random holdout test sets.
# Show overlapping test rows and score variation.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

# Create a small deterministic regression dataset.
features, target = make_regression(
    n_samples=30, n_features=3, noise=12.0, random_state=42
)

# Validate the basic shape before splitting.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# ShuffleSplit makes fresh random holdouts each time.
splitter = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
model = LinearRegression()

# Cross-validation scores estimate performance across random partitions.
scores = cross_val_score(model, features, target, cv=splitter, scoring="r2")

# Inspect which row numbers appear in each test set.
test_sets = []
for train_index, test_index in splitter.split(features):
    test_sets.append(sorted(test_index.tolist()))

# Count how often each row was selected for testing.
test_counts = np.zeros(features.shape[0], dtype=int)
for test_set in test_sets:
    test_counts[test_set] += 1

print("scikit-learn version:", sklearn.__version__)
print("ShuffleSplit settings: 5 splits, 20% test rows each time")
print("Test rows in split 1:", test_sets[0])
print("Test rows in split 2:", test_sets[1])
print("Rows tested more than once:", int(np.sum(test_counts > 1)))
print("Rows never tested:", int(np.sum(test_counts == 0)))
print("R2 scores by split:", np.round(scores, 3).tolist())
print("Mean R2 across random splits:", round(float(np.mean(scores)), 3))



## **3. Special Splitters**

### **3.1. Leave Out Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_03_01.jpg?v=1784032663" width="250">



>* Hold out tiny test sets each fold
>* Trace each prediction to its observation

>* Similar folds estimate near-full-data performance
>* Many fits reveal difficult held-out cases

>* Hold out small groups to test sensitivity
>* Link each prediction to specific observations



In [ ]:
#@title Python Code - Leave Out Methods

# Demonstrate leave-one-out cross-validation fold assignments.
# Each sample becomes one held-out test case.
# Predictions are matched back to original rows.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

# Create a tiny regression dataset for easy inspection.
features, target = make_regression(
    n_samples=8, n_features=2, noise=8.0, random_state=42
)

# Validate that leave-one-out will create one fold per row.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Build the splitter and a simple regression model.
splitter = LeaveOneOut()
model = LinearRegression()

# Inspect the first three fold assignments.
fold_summaries = []
for fold_number, (train_index, test_index) in enumerate(splitter.split(features), 1):
    if fold_number <= 3:
        fold_summaries.append((fold_number, len(train_index), int(test_index[0])))

# Get one out-of-sample prediction for every original row.
predictions = cross_val_predict(model, features, target, cv=splitter)
absolute_errors = np.abs(target - predictions)

# Show a compact table linking rows to held-out predictions.
results = pd.DataFrame(
    {
        "row": np.arange(len(target)),
        "actual": np.round(target, 1),
        "predicted": np.round(predictions, 1),
        "abs_error": np.round(absolute_errors, 1),
    }
)

print("scikit-learn version:", sklearn.__version__)
print("First folds as (fold, train_size, held_out_row):", fold_summaries)
print("Leave-one-out creates folds:", splitter.get_n_splits(features))
print(results.head(5).to_string(index=False))
print("Mean absolute error:", round(mean_absolute_error(target, predictions), 1))



### **3.2. Group Based Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_03_02.jpg?v=1784032665" width="250">



>* Keep related records in the same fold
>* Prevent leakage and test new-group generalization

>* Check group assignments, not just fold sizes
>* Expect imbalance when preserving group separation

>* Assess predictions on entirely held-out groups
>* Compare group errors to find weak spots



In [ ]:
#@title Python Code - Group Based Splits

# Demonstrate group based cross validation folds.
# Keep each group entirely in one fold.
# Inspect predictions for held out groups.

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GroupKFold
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

# Build a tiny dataset with repeated observations per group.
groups = np.repeat(np.arange(1, 7), 3)
visit_number = np.tile(np.arange(1, 4), 6)
group_signal = np.repeat([0, 0, 1, 1, 0, 1], 3)

# Create features and labels with simple deterministic patterns.
X = pd.DataFrame({"visit": visit_number, "signal": group_signal})
y = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1])

# Validate that every row has one group label.
if len(X) != len(y) or len(y) != len(groups):
    raise ValueError("Rows, labels, and groups must have matching lengths.")

# GroupKFold holds out whole groups, not individual rows.
splitter = GroupKFold(n_splits=3)
model = LogisticRegression(max_iter=200, random_state=42)

print("scikit-learn version:", sklearn.__version__)
print("Fold | validation groups | shared groups")

# Inspect which groups appear in validation for each fold.
for fold_number, (train_index, valid_index) in enumerate(
    splitter.split(X, y, groups), start=1
):
    train_groups = set(groups[train_index])
    valid_groups = sorted(set(groups[valid_index]))
    shared_groups = sorted(train_groups.intersection(valid_groups))
    print(f"{fold_number} | {valid_groups} | {shared_groups}")

# Out-of-fold predictions are made for rows from unseen groups.
predicted = cross_val_predict(model, X, y, cv=splitter, groups=groups)
accuracy = accuracy_score(y, predicted)

# Summarize prediction quality by group for quick inspection.
summary = pd.DataFrame({"group": groups, "correct": predicted == y})
group_accuracy = summary.groupby("group")["correct"].mean().round(2)

print("Overall out-of-fold accuracy:", round(accuracy, 2))
print("Group accuracies:", group_accuracy.to_dict())



### **3.3. Time Series Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_27/Lecture_A/image_03_03.jpg?v=1784032667" width="250">



>* Respect time order during validation
>* Test future data after past training

>* Expanding windows grow training data over time
>* Check folds match the forecasting horizon

>* Match predictions to their correct future periods
>* Inspect folds for hidden time-based errors



In [ ]:
#@title Python Code - Time Series Splits

# Demonstrate time series cross-validation splits.
# Preserve order to avoid future leakage.
# Inspect folds and out-of-fold predictions.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

# Create a small ordered monthly demand dataset.
months = np.arange(1, 25)
season = 8 * np.sin(2 * np.pi * months / 12)
demand = 100 + 2 * months + season

# Use previous month demand to predict current demand.
features = demand[:-1].reshape(-1, 1)
target = demand[1:]
periods = months[1:]

# Validate that the supervised rows align correctly.
if len(features) != len(target):
    raise ValueError("Feature and target lengths must match.")

# TimeSeriesSplit trains on earlier rows and tests later rows.
splitter = TimeSeriesSplit(n_splits=4, test_size=3)
model = LinearRegression()
predictions = np.full(len(target), np.nan)
fold_rows = []

# Fit one simple model per fold and store test predictions.
for fold, (train_index, test_index) in enumerate(splitter.split(features), start=1):
    model.fit(features[train_index], target[train_index])
    predictions[test_index] = model.predict(features[test_index])
    fold_rows.append([fold, periods[train_index[-1]], periods[test_index[0]], periods[test_index[-1]]])

# Summarize fold timing without printing large arrays.
fold_table = pd.DataFrame(
    fold_rows,
    columns=["fold", "last_train_month", "first_test_month", "last_test_month"],
)

# Score only months that received out-of-fold predictions.
scored = ~np.isnan(predictions)
mae = mean_absolute_error(target[scored], predictions[scored])

print(f"scikit-learn version: {sklearn_version}")
print("Each test block starts after its training block:")
print(fold_table.to_string(index=False))
print(f"Out-of-fold MAE on tested months: {mae:.2f} demand units")

# Plot actual values and time-respecting cross-validated predictions.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(periods, target, marker="o", label="actual demand")
ax.plot(periods[scored], predictions[scored], marker="s", label="CV prediction")
ax.set_title("TimeSeriesSplit predictions stay in chronological order")
ax.set_xlabel("Month")
ax.set_ylabel("Demand units")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**CV Splitters**</font>


In this lecture, you learned to:
- Use scikit-learn cross-validation utilities to estimate model performance. 
- Choose appropriate splitters for classification, regression, grouped, and time-ordered data. 
- Inspect fold assignments and predictions produced by cross-validation. 

In the next Lecture (Lecture B), we will go over 'Evaluation Design'